# Assistiv Systems — Rural Access Vulnerability Index (RAVI) v1.2

Scores all 1,065 Kent & Medway LSOAs for rural access vulnerability.

**Five signals — open data, no API key:**
- Geographic Barriers (IMD 2019 File 7) — road distance to GP, pharmacy, food store — **30%**
- Rural Urban Classification (mySociety/ONS 2021) — Urban → Hamlet/Isolated — **25%**
- Population aged 65+ (Census 2021 Nomis) — **20%**
- No car or van (Census 2021 Nomis) — **15%**
- Day-to-day limited a lot (Census 2021 Nomis) — **10%**

### Run order:
1. **Cell 1** — fetches all data, calculates RAVI, commits `kent-ravi-data.json`
2. **Cell 2** — adds precise lat/lon coordinates and recommits as v1.2

### Before running:
Add `GITHUB_TOKEN` to Colab Secrets (🔑 left sidebar)

---
*MHCLG IMD 2019 · mySociety/ONS RUC 2021 · Census 2021 Nomis · ONS LSOA Centroids*

## Cell 1 — Data Pipeline
Fetches IMD 2019, RUC 2021, Census 2021, calculates RAVI scores, commits JSON.

In [ ]:
"""
Assistiv Systems — RAVI Pipeline v1.1
Fixed: RUC numeric mapping, Nomis dataset IDs, NaN fallbacks
"""

!pip install requests pandas openpyxl --quiet

import requests, pandas as pd, json, base64, io
from datetime import datetime, timezone
from google.colab import userdata

GITHUB_REPO  = "silegrand/assistivagents"
GITHUB_FILE  = "kent-ravi-data.json"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN').split('\n')[0].strip()

KENT_LAD_CODES = [
    'E07000105','E07000106','E07000107','E07000108','E07000109',
    'E07000110','E06000035','E07000111','E07000112','E07000113',
    'E07000114','E07000115','E07000116',
]

WEIGHTS = {
    'geo_barriers': 0.30,
    'ruc_score':    0.25,
    'pct_65plus':   0.20,
    'pct_no_car':   0.15,
    'pct_llti':     0.10,
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 0.001
print(f"Weight check: {sum(WEIGHTS.values()):.2f} \u2713")

LAD_NAMES = {
    'E07000105':'Ashford',      'E07000106':'Canterbury',
    'E07000107':'Dartford',     'E07000108':'Dover',
    'E07000109':'Gravesham',    'E07000110':'Maidstone',
    'E06000035':'Medway',       'E07000111':'Sevenoaks',
    'E07000112':'Folkestone & Hythe', 'E07000113':'Swale',
    'E07000114':'Thanet',       'E07000115':'Tonbridge & Malling',
    'E07000116':'Tunbridge Wells',
}

# England averages — used as fallback if Nomis unavailable
# Source: Census 2021 national headlines
ENG_FALLBACKS = {
    'pct_65plus': 18.4,   # 18.4% aged 65+ nationally (Census 2021)
    'pct_no_car': 25.6,   # 25.6% households no car (Census 2021)
    'pct_llti':   17.8,   # 17.8% day-to-day limited (Census 2021)
}

HEADERS = {"User-Agent": "Mozilla/5.0 AssistivSystems/1.0"}


# ── STEP 1: IMD 2019 GEOGRAPHIC BARRIERS ─────────────────────────────
print("\n\u2500\u2500 Step 1: IMD 2019 Geographic Barriers \u2500\u2500")

IMD_URL = (
    "https://assets.publishing.service.gov.uk/media/5dc407b440f0b6379a7acc8d/"
    "File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv"
)
r = requests.get(IMD_URL, timeout=120, headers=HEADERS)
print(f"  HTTP {r.status_code} | {len(r.content):,} bytes")
if r.status_code != 200:
    raise RuntimeError(
        "IMD download failed. Download File 7 CSV from:\n"
        "https://www.gov.uk/government/statistics/english-indices-of-deprivation-2019\n"
        "Upload to Colab and update IMD_URL to the local path."
    )

imd_raw = pd.read_csv(io.StringIO(r.text))
print(f"  {len(imd_raw):,} rows | {len(imd_raw.columns)} columns")

lsoa_col  = 'LSOA code (2011)'
lsoa_name = 'LSOA name (2011)'
lad_col   = 'Local Authority District code (2019)'
geo_col   = next((c for c in imd_raw.columns if 'Geographical Barriers' in c and 'Score' in c), None)
if not geo_col:
    geo_col = next((c for c in imd_raw.columns if 'Geograph' in c and 'Score' in c), None)
if not geo_col:
    print("Columns:", list(imd_raw.columns)); raise ValueError("Geo barriers column not found")
print(f"  Geo barriers column: {geo_col!r}")

imd_kent = imd_raw[imd_raw[lad_col].isin(KENT_LAD_CODES)][
    [lsoa_col, lsoa_name, lad_col, geo_col]
].copy()
imd_kent.columns = ['lsoa_code','lsoa_name','lad_code','geo_barriers_score']
imd_kent = imd_kent.dropna(subset=['geo_barriers_score']).reset_index(drop=True)
print(f"  Kent LSOAs: {len(imd_kent)}")
print(f"  Score range: {imd_kent.geo_barriers_score.min():.3f} \u2013 {imd_kent.geo_barriers_score.max():.3f}")


# ── STEP 2: ONS 2021 RURAL URBAN CLASSIFICATION ───────────────────────
print("\n\u2500\u2500 Step 2: Rural Urban Classification \u2500\u2500")

# mySociety composite RUC — ukruc-2: 0=Urban, 1=Rural
RUC_URL = "https://raw.githubusercontent.com/mysociety/uk_ruc/main/output/composite_ruc.csv"
r = requests.get(RUC_URL, timeout=30, headers=HEADERS)
print(f"  HTTP {r.status_code} | {len(r.content):,} bytes")

if r.status_code == 200:
    ruc_raw = pd.read_csv(io.StringIO(r.text))
    print(f"  Columns: {list(ruc_raw.columns)}")
    print(f"  ukruc-2 values: {sorted(ruc_raw['ukruc-2'].unique())}")

    # ukruc-2: 0=Urban, 1=Rural (binary)
    # ukruc-3 (if present): 0=Urban, 1=Rural town/village, 2=Hamlet/Isolated
    ruc_col = 'ukruc-3' if 'ukruc-3' in ruc_raw.columns else 'ukruc-2'
    print(f"  Using RUC column: {ruc_col!r} | Values: {sorted(ruc_raw[ruc_col].unique())}")

    # Map numeric codes to labels and scores
    if ruc_col == 'ukruc-3':
        code_map   = {0:'Urban', 1:'Rural town/village', 2:'Hamlet/Isolated'}
        score_map  = {0:0, 1:55, 2:90}
    else:
        # ukruc-2 binary — 0=Urban, 1=Rural
        code_map   = {0:'Urban', 1:'Rural'}
        score_map  = {0:10, 1:70}

    ruc_kent = ruc_raw[ruc_raw['lsoa'].isin(imd_kent.lsoa_code)][['lsoa', ruc_col]].copy()
    ruc_kent.columns = ['lsoa_code', 'ruc_raw']
    ruc_kent['ruc_label'] = ruc_kent.ruc_raw.map(code_map).fillna('Unknown')
    ruc_kent['ruc_score'] = ruc_kent.ruc_raw.map(score_map).fillna(30.0)

    imd_kent = imd_kent.merge(ruc_kent[['lsoa_code','ruc_label','ruc_score']], on='lsoa_code', how='left')
    imd_kent['ruc_label'] = imd_kent['ruc_label'].fillna('Unknown')
    imd_kent['ruc_score'] = imd_kent['ruc_score'].fillna(30.0)

    matched = imd_kent['ruc_label'].ne('Unknown').sum()
    print(f"  Matched: {matched}/{len(imd_kent)} Kent LSOAs")
    print(f"  RUC distribution:\n{imd_kent.ruc_label.value_counts().to_string()}")
else:
    print("  WARNING: RUC unavailable — assigning 30 to all LSOAs")
    imd_kent['ruc_label'] = 'Unknown'
    imd_kent['ruc_score'] = 30.0


# ── STEP 3: CENSUS 2021 VIA NOMIS ────────────────────────────────────
print("\n\u2500\u2500 Step 3: Census 2021 via Nomis \u2500\u2500")

def fetch_nomis_lsoa(dataset_id, extra_params, description, col_name, england_fallback):
    """
    Fetch LSOA-level Census 2021 % from Nomis.
    Returns df with [lsoa_code, col_name] or None on failure.
    TYPE297 = LSOA 2021 boundaries.
    measures=20301 = % (percentage of total population/households).
    """
    base = (
        f"https://www.nomisweb.co.uk/api/v01/dataset/{dataset_id}.data.csv"
        f"?date=latest&geography=TYPE297&measures=20301"
        f"&{extra_params}&select=GEOGRAPHY_CODE,OBS_VALUE"
    )
    print(f"  {description}...")
    try:
        r = requests.get(base, timeout=60, headers=HEADERS)
        print(f"    HTTP {r.status_code} | {len(r.content):,} bytes")
        if r.status_code == 200 and len(r.content) > 500:
            df = pd.read_csv(io.StringIO(r.text))
            df.columns = ['lsoa_code', col_name]
            df = df[df.lsoa_code.isin(imd_kent.lsoa_code)]
            if len(df) > 100:
                # Sum if multiple categories returned (e.g. age bands)
                df = df.groupby('lsoa_code')[col_name].sum().reset_index()
                print(f"    \u2713 {len(df)} LSOAs | Mean: {df[col_name].mean():.1f}%")
                return df
        print(f"    Empty/short response — using England average {england_fallback}%")
    except Exception as e:
        print(f"    Error: {e} — using England average {england_fallback}%")
    return None

# Census 2021 Nomis dataset IDs (confirmed 2024):
# NM_2010_1 = TS008 Sex by age (use for age groups)
# NM_2072_1 = TS045 Car or van availability
# NM_2064_1 = TS046 Disability

# Age 65+ from TS008 — categories 7-12 cover 65+ age groups
age_df = fetch_nomis_lsoa(
    'NM_2010_1',
    'c2021_age_h=7,8,9,10,11,12,13,14,15,16,17,18',
    'Population aged 65+ %',
    'pct_65plus',
    ENG_FALLBACKS['pct_65plus']
)

# No car — TS045 code 1
car_df = fetch_nomis_lsoa(
    'NM_2072_1',
    'c2021_carsvan_4=1',
    'No car access %',
    'pct_no_car',
    ENG_FALLBACKS['pct_no_car']
)

# LLTI / disability limited a lot — TS046 code 1
llti_df = fetch_nomis_lsoa(
    'NM_2064_1',
    'c2021_disability_3=1',
    'Day-to-day limited a lot %',
    'pct_llti',
    ENG_FALLBACKS['pct_llti']
)

# Merge Census — use England average fallback if Nomis unavailable
for df, col in [(age_df,'pct_65plus'), (car_df,'pct_no_car'), (llti_df,'pct_llti')]:
    fallback = ENG_FALLBACKS[col]
    if df is not None:
        imd_kent = imd_kent.merge(df, on='lsoa_code', how='left')
        n_missing = imd_kent[col].isna().sum()
        if n_missing > 0:
            print(f"  Filling {n_missing} missing {col} with England average {fallback}%")
        imd_kent[col] = imd_kent[col].fillna(fallback)
    else:
        print(f"  Using England average {fallback}% for all {col}")
        imd_kent[col] = fallback

print(f"\nDataset ready: {imd_kent.shape[0]} LSOAs")
print(f"  pct_65plus range: {imd_kent.pct_65plus.min():.1f} \u2013 {imd_kent.pct_65plus.max():.1f}%")
print(f"  pct_no_car range: {imd_kent.pct_no_car.min():.1f} \u2013 {imd_kent.pct_no_car.max():.1f}%")
print(f"  pct_llti range:   {imd_kent.pct_llti.min():.1f} \u2013 {imd_kent.pct_llti.max():.1f}%")


# ── STEP 4: NORMALISE AND SCORE ────────────────────────────────────────
print("\n\u2500\u2500 Step 4: Normalise and calculate RAVI \u2500\u2500")

def norm_to_100(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series([50.0] * len(series), index=series.index)
    return ((series - mn) / (mx - mn) * 100).round(1)

imd_kent['geo_barriers_norm'] = norm_to_100(imd_kent.geo_barriers_score)
imd_kent['ruc_norm']          = norm_to_100(imd_kent.ruc_score)
imd_kent['age_norm']          = norm_to_100(imd_kent.pct_65plus)
imd_kent['car_norm']          = norm_to_100(imd_kent.pct_no_car)
imd_kent['llti_norm']         = norm_to_100(imd_kent.pct_llti)

imd_kent['ravi'] = (
    imd_kent.geo_barriers_norm * WEIGHTS['geo_barriers'] +
    imd_kent.ruc_norm          * WEIGHTS['ruc_score']    +
    imd_kent.age_norm          * WEIGHTS['pct_65plus']   +
    imd_kent.car_norm          * WEIGHTS['pct_no_car']   +
    imd_kent.llti_norm         * WEIGHTS['pct_llti']
).round(1)

def ravi_band(s):
    if s >= 70: return 'critical'
    if s >= 55: return 'high'
    if s >= 40: return 'moderate'
    return 'low'

imd_kent['ravi_band'] = imd_kent.ravi.apply(ravi_band)
imd_kent['district']  = imd_kent.lad_code.map(LAD_NAMES)

print(f"RAVI: {imd_kent.ravi.min():.1f} \u2013 {imd_kent.ravi.max():.1f} | Mean: {imd_kent.ravi.mean():.1f}")
print(f"\nRisk bands:\n{imd_kent.ravi_band.value_counts().to_string()}")
print(f"\nTop 20 most vulnerable LSOAs:")
top = imd_kent.nlargest(20,'ravi')[['lsoa_name','district','ravi','ravi_band','ruc_label','geo_barriers_score']]
print(top.to_string(index=False))


# ── STEP 5: DISTRICT SUMMARY ──────────────────────────────────────────
print("\n\u2500\u2500 Step 5: District summary \u2500\u2500")
print(f"{'District':<25} {'Mean':>5}  {'Max':>5}  {'High/Crit':>9}  Top LSOA")
print("-" * 80)
district_summary = {}
for dist, grp in imd_kent.groupby('district'):
    high = grp[grp.ravi >= 55]
    district_summary[dist] = {
        'lsoa_count':      int(len(grp)),
        'mean_ravi':       round(float(grp.ravi.mean()), 1),
        'max_ravi':        round(float(grp.ravi.max()), 1),
        'high_ravi_count': int(len(high)),
        'pct_high':        round(len(high)/len(grp)*100, 1),
        'top_lsoa':        str(grp.nlargest(1,'ravi').iloc[0]['lsoa_name']),
    }
    s = district_summary[dist]
    print(f"  {dist:<23} {s['mean_ravi']:>5.1f}  {s['max_ravi']:>5.1f}  {s['high_ravi_count']:>9}  {s['top_lsoa'][:35]}")


# ── STEP 6: BUILD AND COMMIT JSON ─────────────────────────────────────
print("\n\u2500\u2500 Step 6: Commit JSON \u2500\u2500")

lsoa_list = [{
    'lsoa_code': row.lsoa_code,
    'lsoa_name': str(row.lsoa_name),
    'district':  str(row.get('district','')),
    'lad_code':  row.lad_code,
    'ravi':      round(float(row.ravi), 1),
    'ravi_band': row.ravi_band,
    'signals': {
        'geo_barriers_score': round(float(row.geo_barriers_score), 3),
        'geo_barriers_norm':  round(float(row.geo_barriers_norm), 1),
        'ruc_label':          str(row.get('ruc_label','Unknown')),
        'ruc_score':          round(float(row.get('ruc_score',30)), 1),
        'ruc_norm':           round(float(row.ruc_norm), 1),
        'pct_65plus':         round(float(row.pct_65plus), 1),
        'age_norm':           round(float(row.age_norm), 1),
        'pct_no_car':         round(float(row.pct_no_car), 1),
        'car_norm':           round(float(row.car_norm), 1),
        'pct_llti':           round(float(row.pct_llti), 1),
        'llti_norm':          round(float(row.llti_norm), 1),
    }
} for _, row in imd_kent.iterrows()]

output = {
    'meta': {
        'generated':  datetime.now(timezone.utc).isoformat(),
        'description':'Rural Access Vulnerability Index — Kent & Medway LSOAs',
        'version':    '1.1',
        'lsoa_count': len(lsoa_list),
        'sources': {
            'geo_barriers':'IMD 2019 File 7 — Geographic Barriers sub-domain (MHCLG)',
            'ruc':         'mySociety composite RUC 2021 (urban=0, rural=1)',
            'census':      'Census 2021 via Nomis API or England average fallback',
        },
        'weights': WEIGHTS,
        'census_source': 'real' if age_df is not None else 'england_average_fallback',
        'note': (
            'RAVI identifies LSOAs where older adults may be hidden from FEP '
            'district-level scores due to geographic isolation and poor service access.'
        ),
    },
    'district_summary': district_summary,
    'lsoas': lsoa_list,
}

def commit_file(content, filepath, token, message):
    api  = f"https://api.github.com/repos/{GITHUB_REPO}/contents/{filepath}"
    hdrs = {"Authorization":f"token {token}","Accept":"application/vnd.github.v3+json"}
    b64  = base64.b64encode(json.dumps(content,indent=2).encode()).decode()
    r    = requests.get(api, headers=hdrs)
    sha  = r.json().get("sha") if r.status_code==200 else None
    pay  = {"message":message,"content":b64,"branch":"main"}
    if sha: pay["sha"] = sha
    r = requests.put(api, headers=hdrs, json=pay)
    if r.status_code in (200,201):
        print(f"  \u2713 {filepath}")
        return True
    print(f"  \u2717 {filepath}: {r.status_code} \u2014 {r.json().get('message','')}")
    return False

msg = f"RAVI v1.1 \u2014 {datetime.now(timezone.utc).strftime('%Y-%m-%d')} \u2014 {len(lsoa_list)} Kent LSOAs"
commit_file(output, GITHUB_FILE, GITHUB_TOKEN, msg)
print(f"\nComplete: {len(lsoa_list)} LSOAs | High/critical: {sum(1 for l in lsoa_list if l['ravi_band'] in ['high','critical'])}")


## Cell 2 — Add Coordinates
Fetches ONS 2021 LSOA population-weighted centroids via ArcGIS FeatureServer.
Falls back to district grid approximations if API unavailable.
Recommits `kent-ravi-data.json` as v1.2 with lat/lon for each LSOA.

In [ ]:
# ── ADD COORDINATES TO RAVI JSON ─────────────────────────────────────
# Fetches ONS LSOA 2021 Population Weighted Centroids via ArcGIS FeatureServer
# Queries all England/Wales in batches, filters to Kent LSOAs
# Falls back to district grid approximations if API unavailable
print("\n── Fetching LSOA centroids ──")

CENTROID_BASE = (
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
    "LSOA_PopCentroids_EW_2021_V4/FeatureServer/0/query"
)

all_centroids = []
offset = 0
batch  = 1000
print("  Querying ONS ArcGIS FeatureServer in batches of 1,000...")

while True:
    try:
        r = requests.get(CENTROID_BASE, timeout=30, headers=HEADERS, params={
            'where': '1=1', 'outFields': 'LSOA21CD,LAT,LONG',
            'f': 'json', 'resultOffset': offset, 'resultRecordCount': batch,
            'orderByFields': 'LSOA21CD',
        })
        data = r.json()
        features = data.get('features', [])
        if not features:
            break
        all_centroids.extend(features)
        offset += len(features)
        if offset % 5000 == 0:
            print(f"  {offset:,} fetched...")
        if len(features) < batch:
            break
    except Exception as e:
        print(f"  Error at offset {offset}: {e}")
        break

print(f"  Total centroids: {len(all_centroids):,}")

cent_df = None
if len(all_centroids) > 1000:
    raw_cent = pd.DataFrame([{
        'lsoa_code': f['attributes']['LSOA21CD'],
        'lat':       round(float(f['attributes']['LAT']), 5),
        'lon':       round(float(f['attributes']['LONG']), 5),
    } for f in all_centroids if f['attributes'].get('LAT') and f['attributes'].get('LONG')])

    cent_kent = raw_cent[raw_cent.lsoa_code.isin(imd_kent.lsoa_code)]
    match_pct = len(cent_kent) / len(imd_kent) * 100
    print(f"  Matched: {len(cent_kent)}/{len(imd_kent)} Kent LSOAs ({match_pct:.0f}%)")

    if match_pct >= 50:
        imd_kent = imd_kent.merge(cent_kent, on='lsoa_code', how='left')
        cent_df = True
        print(f"  ✓ Precise centroids added")
    else:
        print(f"  < 50% matched — 2011/2021 boundary differences too large, using fallback")

if cent_df is None or 'lat' not in imd_kent.columns or imd_kent.lat.isna().all():
    print("  Using district grid approximations as fallback")
    import numpy as np
    np.random.seed(42)
    DC = {
        'Ashford':             (51.148, 0.874),
        'Canterbury':          (51.279, 1.076),
        'Dartford':            (51.446, 0.221),
        'Dover':               (51.126, 1.313),
        'Folkestone & Hythe':  (51.083, 1.167),
        'Gravesham':           (51.442, 0.368),
        'Maidstone':           (51.272, 0.523),
        'Medway':              (51.385, 0.523),
        'Sevenoaks':           (51.272, 0.187),
        'Swale':               (51.333, 0.753),
        'Thanet':              (51.358, 1.383),
        'Tonbridge & Malling': (51.197, 0.273),
        'Tunbridge Wells':     (51.132, 0.263),
    }
    dist_count = {}
    lats, lons = [], []
    for _, row in imd_kent.iterrows():
        d   = row.district
        idx = dist_count.get(d, 0)
        dist_count[d] = idx + 1
        dc  = DC.get(d, (51.27, 0.52))
        lats.append(round(dc[0] - 0.07 + (idx // 10) * 0.016, 5))
        lons.append(round(dc[1] - 0.07 + (idx  % 10) * 0.016, 5))
    imd_kent['lat'] = lats
    imd_kent['lon'] = lons

imd_kent['lat'] = imd_kent['lat'].fillna(51.27)
imd_kent['lon'] = imd_kent['lon'].fillna(0.52)

print(f"\nCoordinates ready: {imd_kent.lat.notna().sum()}/{len(imd_kent)} LSOAs")
print(f"Sample:\n{imd_kent[['lsoa_name','lat','lon']].head(3).to_string()}")

# ── REBUILD lsoa_list WITH COORDINATES AND RECOMMIT ──────────────────
lsoa_list = [{
    'lsoa_code': row.lsoa_code,
    'lsoa_name': str(row.lsoa_name),
    'district':  str(row.get('district', '')),
    'lad_code':  row.lad_code,
    'lat':       round(float(row.lat), 5),
    'lon':       round(float(row.lon), 5),
    'ravi':      round(float(row.ravi), 1),
    'ravi_band': row.ravi_band,
    'signals': {
        'geo_barriers_score': round(float(row.geo_barriers_score), 3),
        'geo_barriers_norm':  round(float(row.geo_barriers_norm), 1),
        'ruc_label':          str(row.get('ruc_label', 'Unknown')),
        'ruc_score':          round(float(row.get('ruc_score', 30)), 1),
        'ruc_norm':           round(float(row.ruc_norm), 1),
        'pct_65plus':         round(float(row.pct_65plus), 1),
        'age_norm':           round(float(row.age_norm), 1),
        'pct_no_car':         round(float(row.pct_no_car), 1),
        'car_norm':           round(float(row.car_norm), 1),
        'pct_llti':           round(float(row.pct_llti), 1),
        'llti_norm':          round(float(row.llti_norm), 1),
    }
} for _, row in imd_kent.iterrows()]

output['lsoas']                    = lsoa_list
output['meta']['has_coordinates']  = True
output['meta']['generated']        = datetime.now(timezone.utc).isoformat()
output['meta']['version']          = '1.2'

msg = f"RAVI v1.2 — {datetime.now(timezone.utc).strftime('%Y-%m-%d')} — coordinates added"
print("\nCommitting v1.2 with coordinates...")
commit_file(output, GITHUB_FILE, GITHUB_TOKEN, msg)
print(f"Done — {len(lsoa_list)} LSOAs with lat/lon")
